# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Raja-saab/Flyrank1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The action queue prioritizes content using the existing baseline refresh score from the project pipeline.

The queue is a decision-support tool: a higher score means the page is ranked earlier for human review. Reason codes translate the available signals into simple explanations that a reviewer can understand.

The ranking is not treated as proof that a page will improve after refresh. It is an observed prioritization signal based on the available data.

In [8]:
# =========================================================
# SECTION 1 — Ranked actions + reason codes
# SELF-CONTAINED VERSION
# =========================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Repository paths
# ---------------------------------------------------------

ROOT = Path("/content/Flyrank1")

if not ROOT.exists():
    raise FileNotFoundError(
        f"Repository not found at {ROOT}. "
        "Run the repository clone cell first."
    )

FEATURE_PATH = (
    ROOT / "data" / "processed" / "refresh_feature_vector.csv"
)

BASELINE_PATH = (
    ROOT / "data" / "processed" / "baseline_refresh_queue.csv"
)

OUTPUT_DIR = (
    ROOT / "work" / "outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ---------------------------------------------------------
# 2. Check required files
# ---------------------------------------------------------

assert FEATURE_PATH.exists(), (
    f"Missing feature vector: {FEATURE_PATH}"
)

assert BASELINE_PATH.exists(), (
    f"Missing baseline queue: {BASELINE_PATH}"
)

# ---------------------------------------------------------
# 3. Load data
# ---------------------------------------------------------

df = pd.read_csv(FEATURE_PATH)
baseline = pd.read_csv(BASELINE_PATH)

print("Feature vector rows:", len(df))
print("Baseline queue rows:", len(baseline))

print("\nBaseline columns:")
print(baseline.columns.tolist())

# ---------------------------------------------------------
# 4. Required columns
# ---------------------------------------------------------

required_columns = {
    "content_id",
    "baseline_refresh_score"
}

missing = (
    required_columns -
    set(baseline.columns)
)

assert not missing, (
    f"Missing required baseline columns: {missing}"
)

# ---------------------------------------------------------
# 5. Rank the queue
# ---------------------------------------------------------

queue = (
    baseline
    .copy()
    .sort_values(
        "baseline_refresh_score",
        ascending=False
    )
    .reset_index(drop=True)
)

queue["rank"] = (
    np.arange(len(queue)) + 1
)

# ---------------------------------------------------------
# 6. Reason codes
# ---------------------------------------------------------

queue["reason_code"] = (
    "High refresh-priority score"
)

# Look for trend information in baseline first.
if "trend_pct" in queue.columns:

    queue.loc[
        queue["trend_pct"] < -10,
        "reason_code"
    ] = "Observed decline"

# Otherwise look for trend information in feature vector.
elif "trend_pct" in df.columns:

    trend_lookup = (
        df
        .drop_duplicates("content_id")
        .set_index("content_id")["trend_pct"]
    )

    queue["trend_pct"] = (
        queue["content_id"]
        .map(trend_lookup)
    )

    queue.loc[
        queue["trend_pct"] < -10,
        "reason_code"
    ] = "Observed decline"

# ---------------------------------------------------------
# 7. Recommended action
# ---------------------------------------------------------

queue["recommended_action"] = (
    "Human review before refresh"
)

# ---------------------------------------------------------
# 8. Display top 20
# ---------------------------------------------------------

display_columns = [
    "rank",
    "content_id",
    "baseline_refresh_score",
    "reason_code",
    "recommended_action"
]

display(
    queue[display_columns].head(20)
)

print(
    f"\nRanked queue size: {len(queue):,}"
)

print(
    "\n✓ Section 1 completed successfully."
)

Feature vector rows: 30000
Baseline queue rows: 30000

Baseline columns:
['content_id', 'client_id', 'baseline_rank', 'baseline_refresh_score', 'visibility_score', 'freshness_risk_score', 'position_opportunity_score', 'depth_gap_score', 'reason_codes', 'suggested_action_baseline', 'is_declining_label', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate', 'content_age_days', 'days_since_last_update', 'word_count', 'trend_direction']


,rank,content_id,baseline_refresh_score,reason_code,recommended_action
0,1,content_9532f197bbc8,0.941189,Observed decline,Human review before refresh
1,2,content_4d1fe5b32dc2,0.934889,High refresh-priority score,Human review before refresh
2,3,content_07f2e7a6f38a,0.934080,Observed decline,Human review before refresh
3,4,content_e5ae436f9a16,0.933606,High refresh-priority score,Human review before refresh
4,5,content_3430a8b94511,0.933559,High refresh-priority score,Human review before refresh
5,6,content_cbd93118300b,0.933263,Observed decline,Human review before refresh
6,7,content_9c195417f6ef,0.932991,Observed decline,Human review before refresh
7,8,content_ba2acb4ebd04,0.931623,High refresh-priority score,Human review before refresh
8,9,content_79b25654070a,0.931363,High refresh-priority score,Human review before refresh
9,10,content_adddad39251c,0.931124,High refresh-priority score,Human review before refresh



Ranked queue size: 30,000

✓ Section 1 completed successfully.


## 2. Intended use and limits

### Intended use

The queue is intended for content/SEO reviewers who need to prioritize a manageable set of pages for investigation.

The score can help answer:

> "Which pages should a human review first?"

It should not be interpreted as:

> "Which pages are guaranteed to improve if refreshed?"

### Limits

The queue is based on the observed data and the project's available features. It may become less reliable when the underlying content mix, search environment, measurement process, or client population changes.

The recommendations are decision-support only. A human should inspect the page and its context before taking action.

The queue should not automatically publish, rewrite, delete, redirect, or otherwise change content.

In [9]:
# =========================================================
# SECTION 2 — Intended use and limits
# =========================================================

print("INTENDED USE")
print("=" * 50)

print(
    "Primary user: content/SEO reviewer"
)

print(
    "Purpose: prioritize pages for human investigation"
)

print(
    "Output: ranked decision-support queue"
)

print("\nLIMITS")
print("=" * 50)

limits = [
    "A high score does not guarantee improvement.",
    "The queue is based on observed historical data.",
    "Performance may change when the underlying data distribution changes.",
    "Human review is required before acting.",
    "The queue should not directly automate content changes."
]

for i, item in enumerate(limits, 1):
    print(f"{i}. {item}")

print("\n✓ Intended use and limitations documented.")

INTENDED USE
Primary user: content/SEO reviewer
Purpose: prioritize pages for human investigation
Output: ranked decision-support queue

LIMITS
1. A high score does not guarantee improvement.
2. The queue is based on observed historical data.
3. Performance may change when the underlying data distribution changes.
4. Human review is required before acting.
5. The queue should not directly automate content changes.

✓ Intended use and limitations documented.


## 3. Human review + the no-go list

Before acting on a recommendation, a human reviewer should check:

1. Whether the page is still relevant to the intended search need.
2. Whether the observed decline is recent or part of a longer pattern.
3. Whether the page has important business or editorial context not represented in the model.
4. Whether a refresh is actually appropriate.
5. Whether the proposed change could remove useful or accurate information.

### No-go list

The system should never automatically:

- publish content changes;
- delete pages;
- redirect URLs;
- change canonical URLs;
- remove important factual information;
- make legal, medical, financial, or other high-stakes claims;
- override an expert/editorial decision;
- treat a model score as proof that a page is low quality.

The model is a prioritization aid, not an autonomous content decision-maker.

In [10]:
# =========================================================
# SECTION 3 — Human review and no-go controls
# =========================================================

print("HUMAN REVIEW CHECKLIST")
print("=" * 50)

review_checks = [
    "Confirm the page is still relevant.",
    "Inspect recent performance context.",
    "Check whether the decline is persistent or temporary.",
    "Review business/editorial context.",
    "Confirm that refresh is appropriate.",
    "Check proposed changes for accuracy and usefulness."
]

for i, item in enumerate(review_checks, 1):
    print(f"{i}. {item}")

print("\nNO-GO ACTIONS")
print("=" * 50)

no_go = [
    "Automatic publishing",
    "Automatic deletion",
    "Automatic URL redirects",
    "Automatic canonical changes",
    "Automatic removal of factual content",
    "Autonomous high-stakes claims",
    "Overriding human/editorial judgment"
]

for i, item in enumerate(no_go, 1):
    print(f"{i}. {item}")

print(
    "\n✓ Human review is required before action."
)

HUMAN REVIEW CHECKLIST
1. Confirm the page is still relevant.
2. Inspect recent performance context.
3. Check whether the decline is persistent or temporary.
4. Review business/editorial context.
5. Confirm that refresh is appropriate.
6. Check proposed changes for accuracy and usefulness.

NO-GO ACTIONS
1. Automatic publishing
2. Automatic deletion
3. Automatic URL redirects
4. Automatic canonical changes
5. Automatic removal of factual content
6. Autonomous high-stakes claims
7. Overriding human/editorial judgment

✓ Human review is required before action.


## 4. Monitoring / retrain triggers

The recommendations should be monitored because the underlying search and content environment can change.

I would review the system when:

- Precision@50 falls materially compared with the validation result.
- The declining rate changes substantially.
- The distribution of important input features changes.
- A new client or content population differs materially from the training data.
- The ranking begins producing many false positives.
- The relationship between the ranking score and observed outcomes weakens.

A retraining review should be triggered when performance degradation persists rather than after a single unusual observation.

The thresholds below are operational starting points and should be treated as monitoring rules rather than proven statistical cutoffs.

In [11]:
# =========================================================
# SECTION 4 — Monitoring / retrain triggers
# =========================================================

print("MONITORING PLAN")
print("=" * 50)

monitoring_rules = pd.DataFrame(
    {
        "Signal": [
            "Precision@50",
            "Declining rate",
            "Feature distribution",
            "False-positive rate",
            "Score/outcome relationship"
        ],
        "Trigger": [
            "Material sustained drop from validated result",
            "Substantial change from training/validation period",
            "Material distribution shift",
            "Persistent increase in top-ranked false positives",
            "Observed weakening of ranking usefulness"
        ],
        "Response": [
            "Investigate and consider retraining",
            "Audit data and labels",
            "Investigate population/data drift",
            "Review features and threshold/ranking logic",
            "Re-evaluate model and baseline"
        ]
    }
)

display(monitoring_rules)

print(
    "\nMonitoring is intended to detect stale recommendations "
    "before relying on them for ongoing decision-support."
)

MONITORING PLAN


,Signal,Trigger,Response
0,Precision@50,Material sustained drop from validated result,Investigate and consider retraining
1,Declining rate,Substantial change from training/validation pe...,Audit data and labels
2,Feature distribution,Material distribution shift,Investigate population/data drift
3,False-positive rate,Persistent increase in top-ranked false positives,Review features and threshold/ranking logic
4,Score/outcome relationship,Observed weakening of ranking usefulness,Re-evaluate model and baseline



Monitoring is intended to detect stale recommendations before relying on them for ongoing decision-support.


## 5. Exports for the paper

The final ranked queue is exported to `work/outputs/`.

The exported file contains the ranking, baseline refresh score, reason code, and recommended action. It is intended as a reproducible artifact for the paper and for later review.

The export does not imply that the recommendations are automatically correct; it preserves the measured prioritization output so that the analysis can be inspected and reproduced.

In [12]:
# =========================================================
# SECTION 5 — Export queue
# =========================================================

# ---------------------------------------------------------
# Final export columns
# ---------------------------------------------------------

export_columns = [
    "rank",
    "content_id",
    "baseline_refresh_score",
    "reason_code",
    "recommended_action"
]

export_columns = [
    c
    for c in export_columns
    if c in queue.columns
]

final_queue = queue[export_columns].copy()

# ---------------------------------------------------------
# Export CSV
# ---------------------------------------------------------

QUEUE_OUTPUT = (
    OUTPUT_DIR
    / "content_action_playbook.csv"
)

final_queue.to_csv(
    QUEUE_OUTPUT,
    index=False
)

# ---------------------------------------------------------
# Export summary
# ---------------------------------------------------------

SUMMARY_OUTPUT = (
    OUTPUT_DIR
    / "content_action_playbook_summary.md"
)

summary_text = f"""# Content Action Playbook

## Purpose

Decision-support queue for prioritizing content for human review.

## Queue size

{len(final_queue):,} records.

## Primary ranking signal

Baseline refresh score.

## Human review requirement

The ranking is not an automatic action. A human should review each recommendation before making content changes.

## Limitations

The queue reflects observed historical data and may become stale when the underlying content or search environment changes.

## Monitoring

Monitor Precision@50, declining rate, feature distributions, false-positive rate, and the relationship between ranking scores and observed outcomes.

## Files

- `content_action_playbook.csv`
"""

SUMMARY_OUTPUT.write_text(
    summary_text,
    encoding="utf-8"
)

# ---------------------------------------------------------
# Verify exports
# ---------------------------------------------------------

print("EXPORTS")
print("=" * 50)

print(
    "Queue:",
    QUEUE_OUTPUT
)

print(
    "Summary:",
    SUMMARY_OUTPUT
)

print(
    "\nQueue exists:",
    QUEUE_OUTPUT.exists()
)

print(
    "Summary exists:",
    SUMMARY_OUTPUT.exists()
)

assert QUEUE_OUTPUT.exists()
assert SUMMARY_OUTPUT.exists()

print("\n✓ Export completed successfully.")

display(
    final_queue.head(20)
)

EXPORTS
Queue: /content/Flyrank1/work/outputs/content_action_playbook.csv
Summary: /content/Flyrank1/work/outputs/content_action_playbook_summary.md

Queue exists: True
Summary exists: True

✓ Export completed successfully.


,rank,content_id,baseline_refresh_score,reason_code,recommended_action
0,1,content_9532f197bbc8,0.941189,Observed decline,Human review before refresh
1,2,content_4d1fe5b32dc2,0.934889,High refresh-priority score,Human review before refresh
2,3,content_07f2e7a6f38a,0.934080,Observed decline,Human review before refresh
3,4,content_e5ae436f9a16,0.933606,High refresh-priority score,Human review before refresh
4,5,content_3430a8b94511,0.933559,High refresh-priority score,Human review before refresh
5,6,content_cbd93118300b,0.933263,Observed decline,Human review before refresh
6,7,content_9c195417f6ef,0.932991,Observed decline,Human review before refresh
7,8,content_ba2acb4ebd04,0.931623,High refresh-priority score,Human review before refresh
8,9,content_79b25654070a,0.931363,High refresh-priority score,Human review before refresh
9,10,content_adddad39251c,0.931124,High refresh-priority score,Human review before refresh


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.